# SDXL pilot: evaluation, evaluator freeze and OWLv2 count calibration

Works on the shared SDXL folder (`pilot_sdxl`, 160 images, objects mug / basket / rabbit) through a Drive shortcut named `sdxl_v1`. Runs on commit `363a21c`, the code version the pilot was generated and evaluated with.

1. Evaluate the pilot with OWLv2 (FP32) and 8-bit Qwen2.5-VL-7B, as for SD 1.5.
2. Upload the consensus human review of the 24 exported images (two independent annotators, disagreements resolved together) and freeze the evaluator. The freeze stores agreement between automatic and human labels per constraint type; agreement is computed against the 8-bit Qwen answers.
3. Calibrate OWLv2 object counting: sweep the confidence threshold and NMS IoU on stored raw detections against human object counts, and compare with Qwen's own count answers. This is a CPU-only analysis; the configured evaluator settings are not changed.

Steps 2 and 3 run on a local copy of the pilot because files written through the shared-folder shortcut were not reliably visible to the session. Results are copied to a separate Drive folder, `sdxl_v1/pilot_sdxl_freeze_results`, so the shared pilot folder is not modified.

**Run order:** Runtime → T4 GPU → cells 1–2 → restart session → cells 1, 3, 4 → remaining cells. The project source is not modified.

## 1. Repository

In [1]:
from pathlib import Path
import subprocess, sys, os
REPO_URL = "https://github.com/IzyumovMaxim/cv_project"
REPO = Path("/content/cv_project")
if not (REPO / "pyproject.toml").exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)
os.chdir(REPO)
subprocess.run(["git", "checkout", "363a21c"], check=True)
print("Commit:", subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip())

Commit: 363a21cea0f319d2da36ade8426e731864c8a914


## 2. Dependencies
Restart the session after this cell.

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", "."], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "bitsandbytes==0.47.0"], check=True)
subprocess.run([sys.executable, "-m", "pip", "check"], check=False)

## 3. GPU check

In [ ]:
import torch
assert torch.cuda.is_available(), "Select a T4 GPU runtime"
print(torch.cuda.get_device_name(0), round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1), "GiB")

## 4. Google Drive and helpers
Mount the account that has the `sdxl_v1` shortcut (editor access to the shared folder).

In [2]:
from google.colab import drive
drive.mount("/content/drive")
import json, re
OUTPUTS = Path("/content/drive/MyDrive/sdxl_v1")
assert (OUTPUTS / "pilot_sdxl" / "run.json").exists(), "Shortcut sdxl_v1 to the shared SDXL folder not found"
ROOT = OUTPUTS / "pilot_sdxl"
CONFIG = REPO / "configs" / "pilot_sdxl.yaml"
QUANT = "int8"

RUNNER = Path("/content/eval_quant.py")
RUNNER.write_text('''import sys, json, shutil, datetime
from pathlib import Path
import attribute_interference.storage as st
import attribute_interference.evaluation as ev
import attribute_interference.analysis as an

config, root, mode, quant = sys.argv[1], Path(sys.argv[2]), sys.argv[3], sys.argv[4]
assert mode in {'evaluate', 'analyze'} and quant in {'int8', 'nf4'}
cfg = st.load_config(config)
QDIR = f'qwen_{quant}_records'

_records, _put = st.records, st.put_record
def records(r, kind):
    return _records(r, QDIR if kind == 'qwen_records' else kind)
def put_record(r, kind, key, value):
    return _put(r, QDIR if kind == 'qwen_records' else kind, key, value)
ev.records = an.records = records
ev.put_record = put_record

def skip_freeze(cfg, root, frozen):
    marker = Path(root) / 'PRELIMINARY_UNFROZEN.json'
    if not marker.exists():
        st.atomic_json(marker, {
            'note': 'Evaluated WITHOUT a reviewed pilot evaluator freeze. Preliminary results only; '
                    'evaluator accuracy not validated against human labels.',
            'started_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds')})
ev.ensure_freeze = skip_freeze

def quantized_init(self, cfg, root, torch):
    from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration, BitsAndBytesConfig
    self.torch = torch
    e = cfg['evaluation']
    revision = ev.model_revision(root, 'qwen', e['qwen_checkpoint'], e['qwen_revision'])
    skip = ['visual', 'lm_head']
    if quant == 'int8':
        q = BitsAndBytesConfig(load_in_8bit=True, llm_int8_skip_modules=skip)
    else:
        q = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                               bnb_4bit_compute_dtype=torch.float16, llm_int8_skip_modules=skip)
    self.model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        e['qwen_checkpoint'], revision=revision, torch_dtype=torch.float16, attn_implementation='sdpa',
        device_map={'': 0}, quantization_config=q).eval()
    self.processor = AutoProcessor.from_pretrained(e['qwen_checkpoint'], revision=revision,
                                                   min_pixels=256*28*28, max_pixels=e['max_pixels'])
    self.max_tokens = e['max_new_tokens']
ev.QwenEvaluator.__init__ = quantized_init

try:
    if mode == 'evaluate':
        st.atomic_json(root / f'QWEN_{quant.upper()}_NOTE.json', {
            'note': f'Qwen2.5-VL-7B evaluated with bitsandbytes {quant} quantization (deviation from the '
                    'proposal, which specifies FP16 without quantization). Answers in ' + QDIR + '/.'})
        ev.evaluate(cfg, root, 'qwen')
    else:
        an.analyze(cfg, root)
        dest = root / f'analysis_qwen_{quant}'
        if dest.exists():
            shutil.rmtree(dest)
        shutil.copytree(root / 'analysis', dest)
        shutil.copy(root / 'labels.csv', dest / 'labels.csv')
        (dest / 'NOTE.txt').write_text(
            f'Preliminary results: Qwen quantized ({quant}), no reviewed evaluator freeze.\\n'
            'OWLv2 unchanged (FP32). Do not present as the final frozen evaluation.\\n')
        print('Results copied to', dest)
except (ValueError, RuntimeError, FileNotFoundError, ImportError) as exc:
    print(f'ERROR: {exc}', file=sys.stderr)
    sys.exit(1)
''')

OWL_CMD = """
import sys, attribute_interference.evaluation as ev
from attribute_interference.storage import load_config
ev.ensure_freeze = lambda *a: None
try:
    ev.evaluate(load_config(sys.argv[1]), sys.argv[2], "owl")
except (ValueError, RuntimeError, FileNotFoundError, ImportError) as exc:
    print("ERROR:", exc, file=sys.stderr); sys.exit(1)
"""

def run(command, config, name, *extra):
    subprocess.run([sys.executable, "-m", "attribute_interference.cli", command,
                    "--config", str(REPO / "configs" / (config + ".yaml")),
                    "--output", str(OUTPUTS / name), *map(str, extra)], check=True, cwd=REPO)

def stream(cmd):
    p = subprocess.Popen(cmd, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        m = re.search(r"\[(\d+)/(\d+)\].*: (\w+)$", line.strip())
        if m is None or m.group(3) != "ok" or int(m.group(1)) % 25 == 0 or m.group(1) == m.group(2):
            print(line, end="", flush=True)
    if p.wait():
        raise RuntimeError(f"Command failed with exit code {p.returncode}; see the output above")

def evaluate(backend):
    if backend == "owl":
        stream([sys.executable, "-c", OWL_CMD, str(CONFIG), str(ROOT)])
    else:
        stream([sys.executable, str(RUNNER), str(CONFIG), str(ROOT), "evaluate", QUANT])

def analyze():
    stream([sys.executable, str(RUNNER), str(CONFIG), str(ROOT), "analyze", QUANT])

def progress():
    gen = [json.loads(p.read_text()) for p in (ROOT / "generation_records").glob("*.json")]
    print(f"Generated: {sum(r['status'] == 'ok' for r in gen)} images")
    for role in ["owl", f"qwen_{QUANT}"]:
        recs = [json.loads(p.read_text()) for p in (ROOT / f"{role}_records").glob("*.json")]
        ok = sum(r["status"] == "ok" for r in recs)
        print(f"{role}: evaluated {ok}, errors {len(recs) - ok}")
print("Outputs:", OUTPUTS)

TOOLS = Path("/content/pilot_tools")
TOOLS.mkdir(exist_ok=True)
(TOOLS / "freeze_runner.py").write_text('''import sys, json
from pathlib import Path
import attribute_interference.storage as st
import attribute_interference.evaluation as ev
import attribute_interference.annotation as an

config, root = sys.argv[1], Path(sys.argv[2])
_records = st.records
ev.records = lambda r, kind: _records(r, 'qwen_int8_records' if kind == 'qwen_records' else kind)
cfg = st.load_config(config)
try:
    an.freeze_evaluator(cfg, root)
except (ValueError, RuntimeError, FileNotFoundError, ImportError, KeyError) as exc:
    print(f'ERROR: {exc}', file=sys.stderr)
    sys.exit(1)
freeze = json.loads((root / 'evaluator_freeze.json').read_text())
st.atomic_json(root / 'evaluator_freeze_note.json', {
    'note': 'Agreement computed against 8-bit (bitsandbytes int8) Qwen2.5-VL-7B answers from qwen_int8_records; '
            'OWLv2 FP32 with config threshold/NMS. Human labels: consensus of two independent annotators.'})
print('Agreement with human labels (auto vs consensus):')
for f, a in freeze['agreement'].items():
    acc = a['correct'] / a['total'] if a['total'] else None
    print(f"  {f:6s} correct {a['correct']:2d} / known {a['total']:2d} (reviewed {a['reviewed']:2d})"
          + (f"  -> {acc:.0%}" if acc is not None else ''))
''')
(TOOLS / "owl_tuning.py").write_text('''import csv, json, itertools
from pathlib import Path
from attribute_interference.evaluation import filter_boxes


def tune(root, gt_csv, thresholds=None, nms_values=None):
    root = Path(root)
    gt = {r['image_id']: int(r['human_count']) for r in csv.DictReader(open(gt_csv, encoding='utf-8'))
          if r['human_count'].strip()}
    owl = {}
    for iid in gt:
        p = root / 'owl_records' / f'{iid}.json'
        if p.exists():
            rec = json.loads(p.read_text())
            if rec.get('status') == 'ok':
                owl[iid] = rec
    qwen = {}
    for iid in gt:
        p = root / 'qwen_int8_records' / f'{iid}.json'
        if p.exists():
            rec = json.loads(p.read_text())
            n = rec.get('raw', {}).get('N', {}).get('parsed_count')
            if n is not None:
                qwen[iid] = n
    thresholds = thresholds or [0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.5]
    nms_values = nms_values or [0.3, 0.4, 0.5, 0.6, 0.7]
    rows = []
    for t, n in itertools.product(thresholds, nms_values):
        errs = []
        for iid, rec in owl.items():
            kept = filter_boxes(rec['raw_boxes'], rec['raw_scores'], t, n, rec['width'], rec['height'])
            errs.append(len(kept) - gt[iid])
        if errs:
            rows.append({'threshold': t, 'nms_iou': n, 'images': len(errs),
                         'exact_acc': round(sum(e == 0 for e in errs) / len(errs), 3),
                         'mae': round(sum(abs(e) for e in errs) / len(errs), 3),
                         'mean_signed_err': round(sum(errs) / len(errs), 3)})
    rows.sort(key=lambda r: (-r['exact_acc'], r['mae']))
    q = [(qwen[i], gt[i]) for i in qwen]
    qwen_summary = {'images': len(q),
                    'exact_acc': round(sum(a == b for a, b in q) / len(q), 3) if q else None,
                    'mae': round(sum(abs(a - b) for a, b in q) / len(q), 3) if q else None}
    current = next((r for r in rows if r['threshold'] == 0.2 and r['nms_iou'] == 0.5), None)
    return rows, current, qwen_summary
''')
sys.path.insert(0, str(TOOLS))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Outputs: /content/drive/MyDrive/sdxl_v1


## 5. Pilot status

In [ ]:
def count(sub, pattern="*.json"):
    d = ROOT / sub
    return len(list(d.glob(pattern))) if d.exists() else 0
print("images:", count("images", "*.png"), "/ 160")
print("owl records:", count("owl_records"), "/ 120 (prompts with a count or position factor)")
print("qwen int8 records:", count("qwen_int8_records"), "/ 160")
print("review files:", sorted(p.name for p in (ROOT / "review").iterdir()) if (ROOT / "review").exists() else "none")
print("freeze exists:", (ROOT / "evaluator_freeze.json").exists())

## 6. Pilot evaluation (skip if already complete)
OWLv2 takes a few minutes; 8-bit Qwen roughly 15–25 minutes after the model download.

In [ ]:
evaluate("owl")
evaluate("qwen")
analyze()
progress()

## 7. Pilot quality check (60% single-constraint criterion)

In [ ]:
import pandas as pd
from IPython.display import display
from attribute_interference.storage import read_csv
OUT = ROOT / "analysis_qwen_int8"
singles = pd.DataFrame(read_csv(OUT / "singletons.csv"))
print(singles.to_string())
display(pd.DataFrame(read_csv(OUT / "attributes.csv")))

## 8. Local working copy
Copies the pilot to `/content/pilot_local`, skipping Drive-only Google Sheets files and OWLv2 records (they are recomputed in the next cell).

In [12]:
import shutil, csv
LOCAL = Path("/content/pilot_local")
if LOCAL.exists():
    shutil.rmtree(LOCAL)
shutil.copytree(ROOT, LOCAL, ignore=shutil.ignore_patterns("owl_records_previous", ".offload", "*.gsheet", "owl_records"))
print("images:", len(list((LOCAL / "images").glob("*.png"))),
      "| qwen int8:", len(list((LOCAL / "qwen_int8_records").glob("*.json"))),
      "| selection:", (LOCAL / "review" / "selection.json").exists())

images: 160 | qwen int8: 160 | selection: True


## 9. Consensus human review and OWLv2 on the local copy
Choose `annotations.csv` (consensus labels for the 24 exported images) in the dialog. OWLv2 is re-run on the local copy so that its records and `environment/owl.json` are written next to the review; this takes about 4 minutes.

In [13]:
from google.colab import files
uploaded = files.upload()
(LOCAL / "review" / "annotations.csv").write_bytes(uploaded[next(n for n in uploaded if n.endswith(".csv"))])
if (LOCAL / "owl_records").exists():
    shutil.rmtree(LOCAL / "owl_records")
stream([sys.executable, "-c", OWL_CMD, str(CONFIG), str(LOCAL)])
print("annotations:", (LOCAL / "review" / "annotations.csv").exists(),
      "| owl.json:", (LOCAL / "environment" / "owl.json").exists(),
      "| qwen.json:", (LOCAL / "environment" / "qwen.json").exists())

Saving annotations.csv to annotations (3).csv
owl [25/120] 2c0ec0342a162dc2_s0: ok
owl [50/120] 5c3e06b1db03dbd8_s1: ok
owl [75/120] a4203094001a7166_s0: ok
owl [100/120] cba8a62b054925d6_s1: ok
owl [120/120] dfd17909ebdca18e_s1: ok
annotations: True | owl.json: True | qwen.json: True


## 10. Evaluator freeze and OWLv2 count calibration
Freezes the evaluator and prints agreement with the human consensus per constraint type. Then choose `owl_count_ground_truth_review24.csv` (consensus object counts; two ambiguous images excluded) for the threshold and NMS sweep. Only images with a count or position factor have OWLv2 detections. Results are copied to `sdxl_v1/pilot_sdxl_freeze_results`.

In [14]:
import json, importlib, pandas as pd
stream([sys.executable, str(TOOLS / "freeze_runner.py"), str(CONFIG), str(LOCAL)])
uploaded = files.upload()
gt_path = TOOLS / "gt_counts.csv"
gt_path.write_bytes(uploaded[next(n for n in uploaded if n.endswith(".csv"))])
import owl_tuning
importlib.reload(owl_tuning)
rows, current, qwen_counts = owl_tuning.tune(LOCAL, gt_path)
print("Current setting (threshold 0.2, NMS 0.5):", current)
print("Qwen count answers vs humans:", qwen_counts)
print(pd.DataFrame(rows).head(10).to_string())
pd.DataFrame(rows).to_csv(LOCAL / "owl_count_calibration.csv", index=False)
json.dump({"current": current, "best": rows[0] if rows else None, "qwen": qwen_counts},
          open(LOCAL / "owl_count_calibration_summary.json", "w"), indent=2)
BACKUP = OUTPUTS / "pilot_sdxl_freeze_results"
BACKUP.mkdir(exist_ok=True)
for name in ["evaluator_freeze.json", "evaluator_freeze_note.json", "owl_count_calibration.csv", "owl_count_calibration_summary.json"]:
    shutil.copy(LOCAL / name, BACKUP / name)
shutil.copy(LOCAL / "review" / "annotations.csv", BACKUP / "annotations_consensus.csv")
print("Saved to", BACKUP)

Frozen evaluator: /content/pilot_local/evaluator_freeze.json. Review agreement fields before using it on main.
Agreement with human labels (auto vs consensus):
  C      correct 14 / known 15 (reviewed 15)  -> 93%
  M      correct 10 / known 10 (reviewed 10)  -> 100%
  N      correct 14 / known 14 (reviewed 14)  -> 100%
  O2     correct  8 / known 11 (reviewed 11)  -> 73%
  P      correct  9 / known 10 (reviewed 10)  -> 90%
  object correct 24 / known 24 (reviewed 24)  -> 100%


Saving owl_count_ground_truth_review24.csv to owl_count_ground_truth_review24.csv
Current setting (threshold 0.2, NMS 0.5): {'threshold': 0.2, 'nms_iou': 0.5, 'images': 16, 'exact_acc': 0.812, 'mae': 0.25, 'mean_signed_err': 0.0}
Qwen count answers vs humans: {'images': 12, 'exact_acc': 0.75, 'mae': 0.5}
   threshold  nms_iou  images  exact_acc    mae  mean_signed_err
0       0.25      0.4      16      0.875  0.188           -0.062
1       0.30      0.5      16      0.875  0.250           -0.250
2       0.20      0.4      16      0.812  0.250            0.000
3       0.20      0.5      16      0.812  0.250            0.000
4       0.25      0.3      16      0.812  0.250           -0.125
5       0.25      0.5      16      0.812  0.250            0.000
6       0.30      0.4      16      0.812  0.312           -0.312
7       0.35      0.5      16      0.812  0.312           -0.312
8       0.40      0.5      16      0.812  0.375           -0.375
9       0.20      0.3      16      0.750  0.

## Outputs
In `sdxl_v1/pilot_sdxl_freeze_results`: `evaluator_freeze.json` (agreement per constraint), `evaluator_freeze_note.json`, `annotations_consensus.csv`, `owl_count_calibration.csv`, `owl_count_calibration_summary.json`. Pilot analysis: `pilot_sdxl/analysis_qwen_int8/`.

## Results
| Constraint | Correct / known | Agreement |
|---|---|---|
| C | 14 / 15 | 93% |
| M | 10 / 10 | 100% |
| N | 14 / 14 | 100% |
| O2 | 8 / 11 | 73% |
| P | 9 / 10 | 90% |
| object | 24 / 24 | 100% |

OWLv2 counting on 16 images with human counts: the configured setting (threshold 0.2, NMS IoU 0.5) gives 81% exact counts, MAE 0.25 and mean signed error 0.0. The best grid point (0.25 / 0.4) gains a single image (87.5%), so the configured setting is kept. Qwen count answers on the same images: 75% exact, MAE 0.5.